# Task 4 — Context-Aware RAG Chatbot (LangChain + FAISS)

**DevelopersHub Corporation — Internship Task 4**

Retrieval-Augmented Generation chatbot with **conversational memory**, **FAISS** vector store, and **Hugging Face** embeddings/LLM — deployable via Streamlit + public tunnel in Colab.

> **Optional:** Set `GROQ_API_KEY` or `GOOGLE_API_KEY` in Colab secrets for faster cloud LLMs; default uses a local `flan-t5-small` pipeline (no API key).

## Cell 1: Environment Setup & Installations

In [5]:
# Cell 1 — Environment setup & quiet installs
import subprocess
import sys
from typing import List


def install_packages(packages: List[str]) -> None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-U", *packages],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
    )


# Do NOT pip-install torch on Colab — upgrades often break torchvision and
# sentence-transformers (error: torchvision has no attribute 'extension').
REQUIRED_PACKAGES = [
    "langchain",
    "langchain-classic",
    "langchain-community",
    "langchain-huggingface",
    "langchain-text-splitters",
    "langchain-core",
    "faiss-cpu",
    "sentence-transformers",
    "fastembed",
    "transformers",
    "accelerate",
    "streamlit",
    "pyngrok",
    "langchain-groq",
]

install_packages(REQUIRED_PACKAGES)
print("All dependencies installed successfully.")
print("If Cell 2 still fails: Runtime → Restart session, then rerun Cells 1–2.")
print("embedding_utils.py is optional — Cells 2–4 include inline embedding loaders.")

# Load API keys from Colab Secrets into os.environ (Secrets are NOT auto-loaded)
import os

try:
    from google.colab import userdata

    for secret_name in ("GROQ_API_KEY", "GOOGLE_API_KEY", "NGROK_AUTHTOKEN"):
        try:
            os.environ[secret_name] = userdata.get(secret_name)
            print(f"Loaded secret: {secret_name}")
        except Exception:
            pass
except ImportError:
    print("Not in Colab — set GROQ_API_KEY in the environment manually if needed.")

All dependencies installed successfully.
If Cell 2 still fails: Runtime → Restart session, then rerun Cells 1–2.
embedding_utils.py is optional — Cells 2–4 include inline embedding loaders.
Loaded secret: GROQ_API_KEY


## Cell 2: Document Ingestion & Vector Database Setup

Build a programmatic knowledge base, chunk with `RecursiveCharacterTextSplitter`, embed with `HuggingFaceEmbeddings`, and persist a **FAISS** index locally.

In [6]:
# Cell 2 — Ingestion, chunking, embeddings, FAISS
from __future__ import annotations

import json
import logging
import warnings
from pathlib import Path
from typing import List

warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

try:
    from embedding_utils import get_embeddings
except ImportError:
    import json
    import subprocess
    import sys

    def get_embeddings():  # Colab fallback if embedding_utils.py not uploaded
        model = "sentence-transformers/all-MiniLM-L6-v2"
        try:
            import sentence_transformers  # noqa
            from langchain_huggingface import HuggingFaceEmbeddings
            emb = HuggingFaceEmbeddings(model_name=model)
            Path("embedding_config.json").write_text(json.dumps({"backend": "huggingface", "model": model}))
            return emb
        except Exception:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "torch", "torchvision", "torchaudio", "sentence-transformers"], check=False)
            try:
                import sentence_transformers
                from langchain_huggingface import HuggingFaceEmbeddings
                emb = HuggingFaceEmbeddings(model_name=model)
                Path("embedding_config.json").write_text(json.dumps({"backend": "huggingface", "model": model}))
                return emb
            except Exception:
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastembed"], check=False)
                from langchain_community.embeddings import FastEmbedEmbeddings
                fb = "BAAI/bge-small-en-v1.5"
                Path("embedding_config.json").write_text(json.dumps({"backend": "fastembed", "model": fb}))
                return FastEmbedEmbeddings(model_name=fb)

EMBEDDING_CONFIG = Path("embedding_config.json")

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

KB_DIR = Path("./knowledge_base")
FAISS_INDEX_DIR = Path("./faiss_devhub_index")
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

KB_DIR.mkdir(parents=True, exist_ok=True)


def build_sample_corpus() -> List[Document]:
    """Programmatic DevelopersHub knowledge base (policies + technical docs)."""
    raw_docs = [
        {
            "title": "company_overview.md",
            "text": (
                "DevelopersHub Corporation is a technology company specializing in AI engineering, "
                "cloud solutions, and enterprise software. Founded in 2018, headquarters are in San Francisco. "
                "The company serves clients in fintech, healthcare, and telecommunications. "
                "Mission: democratize production-grade machine learning for every engineering team."
            ),
        },
        {
            "title": "pto_policy.md",
            "text": (
                "Paid Time Off (PTO) Policy: Full-time employees receive 20 PTO days per calendar year. "
                "PTO accrues monthly and may be carried over up to 5 days into the next year. "
                "Requests must be submitted in the HR portal at least 10 business days in advance for trips "
                "longer than 3 days. Unused PTO is not paid out upon voluntary resignation unless required by local law."
            ),
        },
        {
            "title": "remote_work.md",
            "text": (
                "Remote Work Policy: DevelopersHub supports hybrid and fully remote roles. "
                "Employees may work remotely up to 3 days per week without manager approval. "
                "Fully remote employees must be available core hours 10:00–16:00 in their local timezone. "
                "Company provides a $500 annual stipend for home office equipment."
            ),
        },
        {
            "title": "security_policy.md",
            "text": (
                "Information Security: All source code must be stored in approved Git repositories with branch protection. "
                "Multi-factor authentication (MFA) is mandatory for VPN, email, and cloud consoles. "
                "Customer data must never be copied to personal devices. Report security incidents within 1 hour to security@developershub.com."
            ),
        },
        {
            "title": "ml_platform.md",
            "text": (
                "ML Platform Documentation: The internal platform 'DevHub MLOps' supports experiment tracking, "
                "model registry, and batch inference. Standard Python version is 3.10. Training jobs run on Kubernetes "
                "GPU nodes (T4/A10). All production models require approval from the Model Review Board and must include "
                "a model card describing data lineage, metrics, and bias evaluation."
            ),
        },
        {
            "title": "internship_program.md",
            "text": (
                "Internship Program: DevelopersHub runs a 12-week AI Engineering internship. Tasks include NLP classification, "
                "tabular ML pipelines, multimodal deep learning, and RAG chatbots. Interns are assigned a mentor, "
                "submit weekly demos, and present a capstone in week 12. Performance is evaluated on code quality, "
                "documentation, and deployment readiness."
            ),
        },
        {
            "title": "support_sla.md",
            "text": (
                "Customer Support SLA: Enterprise tier receives P1 response within 1 hour, 24/7. "
                "Business tier P1 response within 4 hours on business days. P2 issues are acknowledged within 1 business day. "
                "Status page updates are posted for any incident affecting more than 5% of API traffic."
            ),
        },
        {
            "title": "api_guidelines.md",
            "text": (
                "Public API Guidelines: REST APIs use OAuth 2.0 client credentials. Rate limit default is 1000 requests/minute "
                "per API key. Breaking changes require 90-day deprecation notice. All endpoints return JSON with standard "
                "error envelopes: code, message, and request_id for tracing."
            ),
        },
    ]

    documents: List[Document] = []
    for item in raw_docs:
        file_path = KB_DIR / item["title"]
        file_path.write_text(item["text"], encoding="utf-8")
        documents.append(
            Document(page_content=item["text"], metadata={"source": str(file_path), "title": item["title"]})
        )
    logger.info("Wrote %s knowledge-base files to %s", len(documents), KB_DIR)
    return documents


documents = build_sample_corpus()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = text_splitter.split_documents(documents)
logger.info("Split %s documents into %s chunks (size=%s, overlap=%s)", len(documents), len(chunks), CHUNK_SIZE, CHUNK_OVERLAP)

embeddings = get_embeddings()  # HuggingFaceEmbeddings or FastEmbed fallback
if not EMBEDDING_CONFIG.exists():
    EMBEDDING_CONFIG.write_text(
        json.dumps({"backend": "huggingface", "model": EMBEDDING_MODEL}),
        encoding="utf-8",
    )
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(str(FAISS_INDEX_DIR))

print(f"FAISS index saved to: {FAISS_INDEX_DIR.resolve()}")
print(f"Sample chunk ({len(chunks[0].page_content)} chars):\n{chunks[0].page_content[:200]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index saved to: /content/faiss_devhub_index
Sample chunk (333 chars):
DevelopersHub Corporation is a technology company specializing in AI engineering, cloud solutions, and enterprise software. Founded in 2018, headquarters are in San Francisco. The company serves clien...


## Cell 3: Building the Conversational RAG Chain

Configure LLM (local Hugging Face pipeline or optional Groq/Gemini), `ConversationBufferMemory`, and `ConversationalRetrievalChain`. Run sequential test queries.

In [8]:
# Cell 3 — Conversational RAG chain + memory + tests
import json
import logging
import os

# Colab: load Groq key from Secrets (skip if you already ran Cell 1 after adding the secret)
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("Groq API key loaded from Colab Secrets.")
except Exception as exc:
    print("Groq secret not loaded:", exc)
from pathlib import Path
from typing import Any, Dict, List, Tuple

import torch
from langchain_community.llms import HuggingFacePipeline
from langchain_community.vectorstores import FAISS

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

FAISS_INDEX_DIR = Path("./faiss_devhub_index")
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_CONFIG = Path("embedding_config.json")


def load_embeddings():
    """Reload embeddings matching Cell 2 (reads embedding_config.json)."""
    if EMBEDDING_CONFIG.exists():
        cfg = json.loads(EMBEDDING_CONFIG.read_text(encoding="utf-8"))
        backend = cfg.get("backend", "huggingface")
        model = cfg.get("model", EMBEDDING_MODEL)
        if backend == "huggingface":
            from langchain_huggingface import HuggingFaceEmbeddings
            return HuggingFaceEmbeddings(model_name=model)
        from langchain_community.embeddings import FastEmbedEmbeddings
        return FastEmbedEmbeddings(model_name=model)
    if "get_embeddings" in globals():
        return get_embeddings()
    raise FileNotFoundError(
        "Run Cell 2 first (creates faiss_devhub_index/ and embedding_config.json)."
    )

# LangChain 1.x: legacy chains live in langchain-classic (no ConversationBufferMemory — it stores str history and breaks Q2+)
try:
    from langchain.chains import ConversationalRetrievalChain
except ModuleNotFoundError:
    from langchain_classic.chains import ConversationalRetrievalChain

print("Using ConversationalRetrievalChain from",
      ConversationalRetrievalChain.__module__)
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

DEVICE = 0 if torch.cuda.is_available() else -1
LOCAL_LLM_MODEL = "google/flan-t5-small"
FALLBACK_LLM_MODEL = "distilgpt2"
RETRIEVER_K = 3


def build_local_hf_pipeline(model_id: str):
    """Build HF pipeline — Colab transformers 5.x removed text2text-generation task name."""
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
    device_arg = DEVICE if DEVICE >= 0 else "cpu"

    attempts = [
        {"task": "summarization", "model": model, "tokenizer": tokenizer},
        {"model": model, "tokenizer": tokenizer},
    ]
    for kwargs in attempts:
        try:
            return pipeline(
                max_new_tokens=256,
                device=device_arg,
                **kwargs,
            )
        except (KeyError, ValueError, TypeError) as exc:
            logger.warning("Pipeline attempt failed (%s): %s", kwargs.get("task", "auto"), exc)

    logger.info("Falling back to %s (text-generation)", FALLBACK_LLM_MODEL)
    from transformers import AutoModelForCausalLM

    fb_tok = AutoTokenizer.from_pretrained(FALLBACK_LLM_MODEL)
    fb_model = AutoModelForCausalLM.from_pretrained(FALLBACK_LLM_MODEL)
    if fb_tok.pad_token is None:
        fb_tok.pad_token = fb_tok.eos_token
    return pipeline(
        "text-generation",
        model=fb_model,
        tokenizer=fb_tok,
        max_new_tokens=256,
        device=device_arg,
    )


def build_llm():
    """Prefer free API LLMs if keys exist; otherwise local HuggingFace pipeline."""
    groq_key = os.environ.get("GROQ_API_KEY", "").strip()
    google_key = os.environ.get("GOOGLE_API_KEY", "").strip()

    if groq_key:
        try:
            from langchain_groq import ChatGroq

            logger.info("Using Groq LLM (llama-3.1-8b-instant)")
            return ChatGroq(model="llama-3.1-8b-instant", temperature=0.2, groq_api_key=groq_key)
        except Exception as exc:
            logger.warning("Groq init failed: %s — falling back to local model", exc)

    if google_key:
        try:
            from langchain_google_genai import ChatGoogleGenerativeAI

            logger.info("Using Google Gemini LLM")
            return ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.2, google_api_key=google_key)
        except Exception as exc:
            logger.warning("Gemini init failed: %s — falling back to local model", exc)

    logger.info("Using local HuggingFacePipeline: %s", LOCAL_LLM_MODEL)
    gen_pipeline = build_local_hf_pipeline(LOCAL_LLM_MODEL)
    return HuggingFacePipeline(pipeline=gen_pipeline)


def build_conversational_chain(vectorstore: FAISS, llm) -> ConversationalRetrievalChain:
    retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVER_K})
    return ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        return_source_documents=True,
        verbose=True,
    )


def query_chain(
    chain: ConversationalRetrievalChain,
    question: str,
    chat_history: List[Tuple[str, str]] | None = None,
) -> Dict[str, Any]:
    """Query RAG chain; logs retrieved context for transparency."""
    chat_history = chat_history or []
    logger.info("User question: %s | prior turns: %s", question, len(chat_history))
    result = chain.invoke({"question": question, "chat_history": chat_history})

    sources = result.get("source_documents", [])
    logger.info("Retrieved %s source chunks for context", len(sources))
    for i, doc in enumerate(sources, start=1):
        preview = doc.page_content[:120].replace("\n", " ")
        logger.info("  Source %s [%s]: %s...", i, doc.metadata.get("title", "?"), preview)

    return result


# Load index + build chain
embeddings = load_embeddings()
vectorstore = FAISS.load_local(
    str(FAISS_INDEX_DIR),
    embeddings,
    allow_dangerous_deserialization=True,
)
llm = build_llm()
rag_chain = build_conversational_chain(vectorstore, llm)

# Manual (human, ai) tuples — conversational memory for langchain_classic
chat_history: List[Tuple[str, str]] = []

q1 = "How many PTO days do full-time DevelopersHub employees get per year?"
r1 = query_chain(rag_chain, q1, chat_history)
print("\nQ1:", q1)
print("A1:", r1["answer"])
chat_history.append((q1, r1["answer"]))

q2 = "Can I carry any unused days to next year?"
r2 = query_chain(rag_chain, q2, chat_history)
print("\nQ2:", q2)
print("A2:", r2["answer"])
chat_history.append((q2, r2["answer"]))

q3 = "What is the name of the internal ML platform mentioned in the docs?"
r3 = query_chain(rag_chain, q3, chat_history)
print("\nQ3:", q3)
print("A3:", r3["answer"])

Groq API key loaded from Colab Secrets.
Using ConversationalRetrievalChain from langchain_classic.chains.conversational_retrieval.base


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
Paid Time Off (PTO) Policy: Full-time employees receive 20 PTO days per calendar year. PTO accrues monthly and may be carried over up to 5 days into the next year. Requests must be submitted in the HR portal at least 10 business days in advance for trips longer than 3 days. Unused PTO is not paid out upon voluntary resignation unless required by local law.

Remote Work Policy: DevelopersHub supports hybrid and fully remote roles. Employees may work remotely up to 3 days per week without manager approval. Fully remote employees must be available core hours 10:00–16:00 in their local timezone. Company provides a $500 annual stipend for home office equipment.

DevelopersHub Corporation is a technology co

## Cell 4: Writing the Chatbot Streamlit Application File

ChatGPT-style UI with `st.chat_message`, session-state history, and visible source snippets.

In [9]:
%%writefile app.py
"""DevelopersHub RAG chatbot — Streamlit frontend."""

from __future__ import annotations

import logging
import os
from pathlib import Path
from typing import Any, Dict, List, Tuple

import streamlit as st
import torch
from langchain_community.llms import HuggingFacePipeline
from langchain_community.vectorstores import FAISS

try:
    from langchain.chains import ConversationalRetrievalChain
except ModuleNotFoundError:
    from langchain_classic.chains import ConversationalRetrievalChain
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

import json

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

FAISS_INDEX_DIR = Path("./faiss_devhub_index")
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_CONFIG = Path("embedding_config.json")
LOCAL_LLM_MODEL = "google/flan-t5-small"
RETRIEVER_K = 3


def load_embeddings():
    if EMBEDDING_CONFIG.exists():
        cfg = json.loads(EMBEDDING_CONFIG.read_text(encoding="utf-8"))
        if cfg.get("backend") == "fastembed":
            from langchain_community.embeddings import FastEmbedEmbeddings
            return FastEmbedEmbeddings(model_name=cfg["model"])
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(model_name=cfg.get("model", EMBEDDING_MODEL))
    from langchain_huggingface import HuggingFaceEmbeddings
    return HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)


@st.cache_resource(show_spinner="Loading embeddings & FAISS index…")
def load_vectorstore() -> FAISS:
    if not FAISS_INDEX_DIR.exists():
        raise FileNotFoundError(
            f"Missing {FAISS_INDEX_DIR}. Run notebook Cell 2 first."
        )
    embeddings = load_embeddings()
    return FAISS.load_local(
        str(FAISS_INDEX_DIR),
        embeddings,
        allow_dangerous_deserialization=True,
    )


@st.cache_resource(show_spinner="Loading LLM & RAG chain…")
def load_rag_chain() -> ConversationalRetrievalChain:
    vectorstore = load_vectorstore()
    retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVER_K})

    groq_key = os.environ.get("GROQ_API_KEY", "").strip()
    google_key = os.environ.get("GOOGLE_API_KEY", "").strip()

    if groq_key:
        try:
            from langchain_groq import ChatGroq

            llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2, groq_api_key=groq_key)
        except Exception:
            llm = None
    else:
        llm = None

    if llm is None and google_key:
        try:
            from langchain_google_genai import ChatGoogleGenerativeAI

            llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.2, google_api_key=google_key)
        except Exception:
            llm = None

    if llm is None:
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline as hf_pipeline

        def _build_local_pipeline(model_id: str):
            tok = AutoTokenizer.from_pretrained(model_id)
            mdl = AutoModelForSeq2SeqLM.from_pretrained(model_id)
            dev = 0 if torch.cuda.is_available() else "cpu"
            for kwargs in [{"task": "summarization"}, {}]:
                try:
                    return hf_pipeline(model=mdl, tokenizer=tok, max_new_tokens=256, device=dev, **kwargs)
                except (KeyError, ValueError, TypeError):
                    continue
            from transformers import AutoModelForCausalLM
            fb = "distilgpt2"
            ft = AutoTokenizer.from_pretrained(fb)
            fm = AutoModelForCausalLM.from_pretrained(fb)
            if ft.pad_token is None:
                ft.pad_token = ft.eos_token
            return hf_pipeline("text-generation", model=fm, tokenizer=ft, max_new_tokens=256, device=dev)

        llm = HuggingFacePipeline(pipeline=_build_local_pipeline(LOCAL_LLM_MODEL))

    return ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        return_source_documents=True,
        verbose=False,
    )


def format_sources(source_documents: List[Any]) -> str:
    lines = []
    for i, doc in enumerate(source_documents, start=1):
        title = doc.metadata.get("title", doc.metadata.get("source", "unknown"))
        snippet = doc.page_content[:300].replace("\n", " ")
        lines.append(f"**[{i}] {title}**\n{snippet}…")
    return "\n\n".join(lines) if lines else "_No sources returned._"


def run_rag_turn(chain: ConversationalRetrievalChain, question: str, chat_history: List[Tuple[str, str]]) -> Dict[str, Any]:
    logger.info("RAG query: %s | history_len=%s", question, len(chat_history))
    return chain.invoke({"question": question, "chat_history": chat_history})


st.set_page_config(page_title="DevelopersHub RAG Assistant", page_icon="💬", layout="centered")
st.title("💬 DevelopersHub RAG Assistant")
st.caption("Retrieval-Augmented Generation • FAISS • LangChain • Conversational memory")

if "messages" not in st.session_state:
    st.session_state.messages = []
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

try:
    chain = load_rag_chain()
except FileNotFoundError as exc:
    st.error(str(exc))
    st.stop()

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if msg["role"] == "assistant" and msg.get("sources_md"):
            with st.expander("📎 Retrieved context"):
                st.markdown(msg["sources_md"])

user_prompt = st.chat_input("Ask about DevelopersHub policies or documentation…")

if user_prompt:
    st.session_state.messages.append({"role": "user", "content": user_prompt})
    with st.chat_message("user"):
        st.markdown(user_prompt)

    with st.chat_message("assistant"):
        with st.spinner("Retrieving context & generating answer…"):
            try:
                result = run_rag_turn(chain, user_prompt, st.session_state.chat_history)
                answer = result.get("answer", "Sorry, I could not generate a response.")
                sources_md = format_sources(result.get("source_documents", []))
            except Exception as exc:
                answer = f"Error: {exc}"
                sources_md = ""

        st.markdown(answer)
        if sources_md:
            with st.expander("📎 Retrieved context"):
                st.markdown(sources_md)

    st.session_state.chat_history.append((user_prompt, answer))
    st.session_state.messages.append(
        {"role": "assistant", "content": answer, "sources_md": sources_md}
    )

Writing app.py


## Cell 5: Background App Execution & Tunnel Deployment

Start Streamlit on port **8501** and expose via **localtunnel** (default) or ngrok.

In [10]:
# Cell 5 — Streamlit + public tunnel
import os
import re
import subprocess
import sys
import time
from pathlib import Path

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("Groq key passed to Streamlit environment.")
except Exception:
    pass

STREAMLIT_PORT = 8501
APP_PATH = Path("app.py")
TUNNEL_METHOD = "localtunnel"  # or "ngrok"

if not APP_PATH.exists():
    raise FileNotFoundError("app.py not found. Run Cell 4 first.")
if not Path("./faiss_devhub_index").exists():
    raise FileNotFoundError("./faiss_devhub_index not found. Run Cell 2 first.")

subprocess.run(["pkill", "-f", f"streamlit run.*{STREAMLIT_PORT}"], check=False)
time.sleep(1)

env = os.environ.copy()
env["STREAMLIT_SERVER_HEADLESS"] = "true"
env["STREAMLIT_BROWSER_GATHER_USAGE_STATS"] = "false"

subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", str(APP_PATH),
        "--server.port", str(STREAMLIT_PORT),
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.fileWatcherType", "none",
    ],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print("Starting Streamlit…")
time.sleep(8)

if TUNNEL_METHOD == "ngrok":
    from pyngrok import ngrok

    token = os.environ.get("NGROK_AUTHTOKEN", "").strip()
    if token:
        ngrok.set_auth_token(token)
    public_url = ngrok.connect(STREAMLIT_PORT, bind_tls=True).public_url
else:
    proc = subprocess.Popen(
        ["npx", "--yes", "localtunnel", "--port", str(STREAMLIT_PORT)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    public_url = None
    url_re = re.compile(r"https://[a-zA-Z0-9-]+\.loca\.lt")
    deadline = time.time() + 90
    while time.time() < deadline:
        line = proc.stdout.readline() if proc.stdout else ""
        if line:
            print(line.rstrip())
            m = url_re.search(line)
            if m:
                public_url = m.group(0)
                break

print("\n" + "=" * 60)
if public_url:
    print("PUBLIC CHATBOT URL:")
    print(public_url)
    print("=" * 60)
    print("Keep this cell running. Click 'Continue' if localtunnel prompts once.")
else:
    print(f"Tunnel failed. Local only: http://127.0.0.1:{STREAMLIT_PORT}")

Groq key passed to Streamlit environment.
Starting Streamlit…
your url is: https://slick-feet-glow.loca.lt

PUBLIC CHATBOT URL:
https://slick-feet-glow.loca.lt
Keep this cell running. Click 'Continue' if localtunnel prompts once.


---
## Deploy only (index + app already built)

If Cells 1–4 finished, run **only this cell** to launch Streamlit + tunnel (~2 min). **No re-indexing.**

In [ ]:
# DEPLOY ONLY
import os, re, subprocess, sys, time
from pathlib import Path
PORT = 8501
assert Path("app.py").exists() and Path("faiss_devhub_index").exists()
subprocess.run(["pkill", "-f", "streamlit"], check=False)
time.sleep(2)
env = os.environ.copy()
env["STREAMLIT_SERVER_HEADLESS"] = "true"
subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", str(PORT), "--server.address", "0.0.0.0",
     "--server.fileWatcherType", "none"],
    env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(8)
proc = subprocess.Popen(["npx", "--yes", "localtunnel", "--port", str(PORT)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for _ in range(90):
    line = proc.stdout.readline()
    if line:
        print(line.rstrip())
        m = re.search(r"https://[a-zA-Z0-9-]+\.loca\.lt", line)
        if m:
            print("\n>>> OPEN:", m.group(0))
            break